# Talk map (Leaflet cluster map of talk locations)

A standalone, self-contained version that needs **no geocoding service**. City coordinates are hardcoded, so it never hits Nominatim and can't be rate-limited (no more 429s).

Each marker is one city; clicking it lists every talk you gave there. Running the last cell writes `talkmap.html`, which you can open directly or embed in a page with an `<iframe>`.

To add a talk, append a `(event, city, date)` tuple to `talks`. If the city is new, add one line to `coords` with its `(latitude, longitude)`.

In [2]:
!pip install folium

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
# !pip install folium
import folium
from folium.plugins import MarkerCluster
from collections import defaultdict
import html

In [4]:
# City coordinates (latitude, longitude) -- city centres, plenty accurate at map scale.
coords = {
    "Vienna":               (48.2082,  16.3738),
    "Trento":               (46.0667,  11.1167),
    "Baltimore":            (39.2904, -76.6122),
    "Bologna":              (44.4949,  11.3426),
    "Cagliari":             (39.2238,   9.1217),
    "Santiago":             (-33.4489, -70.6693),
    "L'Aquila":             (42.3498,  13.3995),
    "Milan":                (45.4642,   9.1900),
    "Paris":                (48.8566,   2.3522),
    "Nice":                 (43.7102,   7.2620),
    "Trieste":              (45.6495,  13.7768),
    "Geneva":               (46.2044,   6.1432),
    "Pisa":                 (43.7160,  10.3966),
    "Rome":                 (41.9028,  12.4964),
    "Sexten":               (46.7000,  12.3500),
    "Opatija":              (45.3378,  14.3053),
    "Orosei":               (40.3786,   9.6989),
    "Providence":           (41.8240, -71.4128),
    "Maastricht":           (50.8514,   5.6910),
    "Leiden":               (52.1601,   4.4970),
    "Liège":                (50.6326,   5.5797),
    "Aspen":                (39.1911, -106.8175),
    "Valencia":             (39.4699,  -0.3763),
    "Melbourne":            (-37.8136, 144.9631),
    "Padova":               (45.4064,  11.8768),
    "São José dos Campos":  (-23.1896, -45.8841),
    "La Thuile":            (45.7117,   6.9536),
    "Innsbruck":            (47.2692,  11.4041),
    "New York":             (40.7128, -74.0060),
}

In [5]:
# (event, city, date) -- one entry per talk. Virtual meetings are omitted.
talks = [
    # Invited colloquia and seminars
    ("Invited seminar, Marietta Blau Institute",                 "Vienna",     "23 Jan 2026"),
    ("Invited seminar, TIFPA",                                   "Trento",     "4 Dec 2025"),
    ("Invited seminar, Johns Hopkins University",                "Baltimore",  "10 Jun 2025"),
    ("Invited talk, GW-BO meeting, Univ. of Bologna",            "Bologna",    "11 Feb 2025"),
    ("Invited seminar, INFN-Cagliari",                           "Cagliari",   "29 Nov 2024"),
    ("Invited seminar, Universidad Andrés Bello",                "Santiago",   "6 Aug 2024"),
    ("Invited seminar, INFN-LNGS",                               "L'Aquila",   "22 Mar 2024"),
    ("Invited seminar, Univ. of Milano-Bicocca",                 "Milan",      "22 Sep 2022"),
    ("Invited talk, High Energy Group, IAP",                     "Paris",      "24 Jun 2021"),
    ("Invited talk, GW group, Observatoire de la Côte d'Azur",   "Nice",       "29 May–2 Jun 2022"),
    ("Invited talk, SISSA Astrophysics Colloquium",              "Trieste",    "13 Apr 2021"),
    # Conferences and workshops
    ("AI for Gravitational Wave workshop, CERN",                 "Geneva",     "5–8 May 2026"),
    ("GRaviCon 2026 (invited, on behalf of LVK)",                "Pisa",       "22–24 Apr 2026"),
    ("ET science workshop for ECRs (invited + panel)",           "Rome",       "18–20 Feb 2026"),
    ("GWFreeride",                                               "Sexten",     "26–30 Jan 2026"),
    ("4th Einstein Telescope annual meeting",                    "Opatija",    "11–14 Nov 2025"),
    ("Deep Space to Deep Earth geophysical workshop",            "Orosei",     "6–10 Oct 2025"),
    ("1st Sardinian GW Science mini-workshop (invited + panel)", "Cagliari",   "24–25 Sep 2025"),
    ("EuCAIFCon 2025",                                           "Cagliari",   "16–20 Jun 2025"),
    ("SciML for GW Astronomy, ICERM (poster)",                   "Providence", "2–6 Jun 2025"),
    ("XV Einstein Telescope Symposium",                          "Bologna",    "26–30 May 2025"),
    ("O4 and beyond, Lorentz Center (talk + panel)",             "Leiden",     "14–18 Oct 2024"),
    ("Annual TEONGRAV meeting, Sapienza",                        "Rome",       "16–20 Sep 2024"),
    ("41st LIAC, Univ. de Liège",                                "Liège",      "15–19 Jul 2024"),
    ("Fundamental Physics in the Era of Big Data & ML",          "Aspen",      "27 May–9 Jul 2024"),
    ("XIV ET Symposium",                                         "Maastricht", "6–10 May 2024"),
    ("Fourth EPS Conference on Gravitation: Black Holes",        "Valencia",   "13–15 Nov 2023"),
    ("GraSP23, Univ. of Pisa (poster)",                          "Pisa",       "24–27 Oct 2023"),
    ("IV Gravi-Gamma-Nu workshop, GSSI",                         "L'Aquila",   "4–6 Oct 2023"),
    ("GWPOPNEXT, Univ. of Milano-Bicocca (panel)",               "Milan",      "10–14 Jul 2023"),
    ("XIII ET Symposium",                                        "Cagliari",   "8–12 May 2023"),
    ("EAS2022, GW & Multi-messenger symposium",                  "Valencia",   "27 Jun–1 Jul 2022"),
    ("Bayesian Deep Learning in Astrophysics",                   "Paris",      "20–24 Jun 2022"),
    ("APS April Meeting 2022",                                   "New York",   "9–12 Apr 2022"),
    ("GWday",                                                    "Padova",     "2 Dec 2021"),
    ("Amaldi14",                                                 "Melbourne",  "19–23 Jul 2021"),
    ("EAS2021, Birth/Life/Death of Black Holes",                 "Leiden",     "28 Jun–2 Jul 2021"),
    ("3rd Workshop on Chemical Abundances in Gaseous Nebulae",   "São José dos Campos", "24–28 May 2021"),
    ("4th GdR Ondes Gravitationnelles assembly",                 "Paris",      "30 Mar–1 Apr 2021"),
    ("55th Rencontres de Moriond (poster)",                      "La Thuile",  "9–11 Mar 2021"),
    ("EAS2020, observed population of GW sources",               "Leiden",     "29 Jun–3 Jul 2020"),
    ("Mock Innsbruck (invited)",                                 "Innsbruck",  "10–13 Mar 2020"),
]

In [6]:
# Group talks by city and check every city has coordinates.
by_city = defaultdict(list)
for event, city, date in talks:
    by_city[city].append((date, event))

missing = [c for c in by_city if c not in coords]
assert not missing, f"Add coordinates for: {missing}"
print(f"{len(talks)} talks across {len(by_city)} cities")

42 talks across 29 cities


In [7]:
# Build the clustered Leaflet map.
m = folium.Map(location=[30, 5], zoom_start=2, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(m)

for city, items in by_city.items():
    lat, lon = coords[city]
    items = sorted(items, reverse=True)  # newest talk first
    rows = "".join(
        f"<li>{html.escape(date)} — {html.escape(event)}</li>"
        for date, event in items
    )
    n = len(items)
    popup_html = (
        f"<b>{html.escape(city)}</b> ({n} talk{'s' if n > 1 else ''})"
        f"<ul style='padding-left:1.1em;margin:4px 0'>{rows}</ul>"
    )
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{city} ({n})",
    ).add_to(cluster)

m.save("talkmap.html")
print("wrote talkmap.html")
m  # also renders inline in the notebook

wrote talkmap.html
